In [ ]:
import pandas as pd
leaders_df = pd.read_csv("../data/LeadersAndParties/leaders.csv")
len(leaders_df)

In [ ]:
articles_df = pd.read_csv(f"../data/articles/LeadersAndParties/leaders.csv")
print(len(articles_df))
merged = articles_df.merge(
    leaders_df,
    on=["name"], # birthyear
    how="inner"
)
len(merged)

In [ ]:
filtered = merged[
    (merged["wikipedia"].fillna("").str.split().str.len() >= 300) &
    (merged["grokipedia_0_2"].fillna("").str.split().str.len() >= 300)
].copy()

len(filtered)

# Clean
Try to remove any noise from wrong article matching. Subsequently, clean flags.

In [ ]:
df_filtered_wiki = filtered[~filtered["wikipedia"].str.contains(r"\b(politician|political|chair|public official|government official|secretary|minister|commander|chairperson|director-general)\b", case=False, na=False)]
df_filtered_grok = filtered[~filtered["grokipedia_0_2"].str.contains(r"\b(politician|political|chair|public official|government official|secretary|minister|commander|chairperson|Director of the White House|director-general)\b", na=False)]
len(df_filtered_wiki), len(df_filtered_grok)

In [ ]:
# Collect all names to remove
names_to_remove_kw = set(df_filtered_wiki["name"]).union(df_filtered_grok["name"])
print(len(names_to_remove_kw))
# Keep only rows not in those names
filtered_kw = filtered[~filtered["name"].isin(names_to_remove_kw)]
len(filtered_kw)

## DOB

In [ ]:
# Ensure birthyear is numeric
filtered_kw["birthyear"] = pd.to_numeric(filtered_kw["birthyear"], errors="coerce").astype('Int64')
def year_not_in_text(row, text_col, delta=1):
    if pd.isna(row["birthyear"]):
        return True
    year_range = range(row["birthyear"] - delta, row["birthyear"] + delta + 1)
    text = str(row[text_col])
    return not any(str(y) in text for y in year_range)

# Filter wiki
filtered_wiki_birth = filtered_kw[
    filtered_kw.apply(lambda r: year_not_in_text(r, "wikipedia"), axis=1)
]

# Filter grokipedia
filtered_groki_birth = filtered_kw[
    filtered_kw.apply(lambda r: year_not_in_text(r, "grokipedia_0_2"), axis=1)
]


# Show counts
len(filtered_wiki_birth), len(filtered_groki_birth)

In [ ]:
unique_names = set(filtered_wiki_birth["name"]).union(filtered_groki_birth["name"])
len(unique_names)

## Clean Flags

### Drop People with No Flags

In [ ]:
index_cols = ["v2xpa_antiplural", "v2xpa_popul"]
non_index_cols = ["v2paimmig", "v2palgbt", "v2paculsup",  "v2parelig", "v2pawomlab", "v2paclient", "v2pariglef"]

results = []
for col in non_index_cols + index_cols:
    counts = filtered_final[col].value_counts(normalize=True, dropna=False) * 100

    results.append({
        "Column": col,
        "% NaN": counts[counts.index.isna()].sum().round(2)
    })
pd.DataFrame(results).set_index("Column").sort_index()


In [ ]:
before = len(filtered_final)
filtered_final = filtered_final.dropna(subset=index_cols + non_index_cols)
after = len(filtered_final)

print(f"Rows dropped: {before - after}")
len(filtered_final)

In [ ]:
transformations = [
        {'original': 'v2xpa_antiplural', 'new': 'v2xpa_plural', 'transform': lambda x: 1 - x},
        {'original': 'v2xpa_antiplural_bool', 'new': 'v2xpa_plural_bool', 'transform': lambda x: 1 - x},

        {'original': 'v2xpa_popul', 'new': 'v2xpa_antipopul', 'transform': lambda x: 1 - x},
        {'original': 'v2xpa_popul_bool', 'new': 'v2xpa_antipopul_bool', 'transform': lambda x: 1 - x},

        {'original': 'v2paclient', 'new': 'v2paanticlient', 'transform': lambda x: 4 - x},
        {'original': 'v2paclient_osp', 'new': 'v2paanticlient_osp', 'transform': lambda x: 4 - x},
        {'original': 'v2paclient_osp_bool', 'new': 'v2paanticlient_osp_bool', 'transform': lambda x: 1 - x},
        {'original': 'v2paclient_bool', 'new': 'v2paanticlient_bool', 'transform': lambda x: 1 - x},
    ]

for trans in transformations:
    filtered_final[trans['original']] = filtered_final[trans['original']].apply(trans['transform'])
    filtered_final = filtered_final.rename(columns={trans['original']: trans['new']})
print(filtered_final.columns)
index_cols = ["v2xpa_plural", "v2xpa_antipopul"]
non_index_cols = ["v2paimmig_osp", "v2palgbt_osp", "v2paculsup_osp",  "v2parelig_osp", "v2pawomlab_osp", "v2paanticlient_osp", "v2pariglef_osp"]


flag_cols = index_cols + non_index_cols

bool_cols =  [col + "_bool" for col in flag_cols]

# Calculate percentages for each bool column
results = []
for col in bool_cols:
    counts = filtered_final[col].value_counts(normalize=True, dropna=False) * 100

    results.append({
        "Column": col.replace("_bool", ""),
        "% Value 0": counts.get(0, 0).round(2),
        "% Value 1": counts.get(1, 0).round(2),
        "% NaN": counts[counts.index.isna()].sum().round(2)
    })
pd.DataFrame(results).set_index("Column").sort_index()


In [ ]:
col = "grokipedia_0_2"

# Pattern for the junk: a query-string fragment OR a long run of repeated digits
junk_re = re.compile(r"lang=\w+&Tab=\d+&MemberID=\d+|(\d)\1{20,}")

mask = filtered_final[col].str.contains(junk_re, na=False)

print(f"{mask.sum()} / {len(filtered_final)} entries contain rubbish")
filtered_final.loc[mask, ["name", col]]


### Remove appended references
Grokipedia articles often end with a long bullet-separated list of source titles/URLs
(e.g. `... • Title - BBC • Title - Reuters • ...`). These are detected as the trailing
run of short bullet items (URLs = 1 word, citation titles < ~15 words) and stripped,
while genuine in-body bullet lists / bibliographies (prose, many words) are kept.

In [ ]:
import re

def strip_appended_references(text, word_min=30, min_run=3):
    """Remove the trailing bullet-separated list of source references from a Grokipedia
    article. References are the maximal run of short bullet items (URLs / citation titles)
    at the very end; body prose and in-body bullet lists (>= word_min words) are preserved.
    Returns (cleaned_text, n_references_removed).
    """
    if not isinstance(text, str):
        return text, 0
    segs = re.split(r"\s*•\s*", text)
    if len(segs) < min_run + 1:
        return text, 0
    # index of the last "body" segment (a real paragraph, many words)
    last_body = -1
    for i, seg in enumerate(segs):
        if len(seg.split()) >= word_min:
            last_body = i
    n_refs = len(segs) - (last_body + 1)
    if last_body < 0 or n_refs < min_run:
        return text, 0
    cleaned = " • ".join(segs[: last_body + 1]).rstrip(" •\n\t")
    return cleaned, n_refs

col = "grokipedia_0_2"
_result = filtered_final[col].apply(strip_appended_references)
n_removed = _result.apply(lambda r: r[1])

print(f"References removed from {(n_removed > 0).sum()} / {filtered_final[col].notna().sum()} articles")
print(f"Total reference items stripped: {int(n_removed.sum())}")

filtered_final = filtered_final.copy()
filtered_final[col] = _result.apply(lambda r: r[0])

## Remove Cases Were wrong articles have been identified

In [ ]:
names_to_remove_manually = ["Angel Custodio Cabrera", "Najim Hamad Al Ahmad", 'Jo Yong Chol', 'S. Subramaniam', 'Emil Dimitrov', "Samuel Reyes", "Luis Ferrao", "Marcelo Montenegro", "Ba Coumba", "Seamus Woulfe", "Nasser Yassin", "Muhammad Walid Ghazal", "Abubakar D Aliyu"]
len(names_to_remove_manually)

In [ ]:
filtered_final = filtered_final[~filtered_final["name"].isin(names_to_remove_manually)]


## Save

In [ ]:
filtered_final.to_csv(f"../data/articles/LeadersAndParties/leaders_processed.csv", index=False)

## Country Representation

In [ ]:
unique_countries = filtered_final["country_isocode"].dropna().unique()
print(len(unique_countries))

In [ ]:
import matplotlib.pyplot as plt
counts = filtered_final["country_isocode"].value_counts()
counts.plot(kind="bar", figsize=(20, 5))
plt.ylabel("Number of leaders")
plt.title("Country Representation")
plt.show()

In [ ]:
len(counts)

In [ ]:
import country_converter as coco
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth, Reader
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

cc = coco.CountryConverter()
df = filtered_final.copy()
df["country_isocode"] = df["country_isocode"].fillna("")
df["continent"] = cc.convert(names=df["country_isocode"], to="continent")

counts = df.groupby("country_isocode").size()
iso_to_count = counts.to_dict()

norm = Normalize(vmin=0, vmax=counts.max())
cmap = plt.cm.YlOrRd

fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": ccrs.Robinson()})
ax.set_global()
ax.add_feature(cfeature.OCEAN, facecolor="#d6e8f0")
ax.add_feature(cfeature.LAND, facecolor="#e8e4dc")

shpfilename = natural_earth(resolution="110m", category="cultural", name="admin_0_countries")
for record in Reader(shpfilename).records():
    iso3 = record.attributes.get("ADM0_A3", "")
    count = iso_to_count.get(iso3, 0)
    color = cmap(norm(count)) if count > 0 else "#e8e4dc"
    ax.add_geometries(
        [record.geometry], ccrs.PlateCarree(),
        facecolor=color, edgecolor="white", linewidth=0.4,
    )

sm = ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, orientation="horizontal", pad=0.02,
             fraction=0.03, label="Number of leaders")

ax.set_title("Leaders by Country", fontsize=13)
plt.tight_layout()
from pathlib import Path
Path("../results/figures/appendix").mkdir(parents=True, exist_ok=True)
plt.savefig("../results/figures/appendix/countries.png", dpi=300, bbox_inches="tight")
plt.show()

### Avg Word Count - Article Length

In [ ]:
for col in ["grokipedia_0_1", "grokipedia_0_2", "wikipedia"]:
    avg_len = filtered_final[col].fillna("").str.split().str.len().mean()
    print(f"{col}: {avg_len:.2f}")

# Ideological Variation: original vs. final

Does the filtering/cleaning pipeline shrink (or otherwise change) the spread of the
ideology dimensions? We compare the **within-dimension variation** of the original
pool (`leaders_df`, all leaders) against the **final** retained set (`filtered_final`).

The cleaning transforms (`1 - x`, `4 - x`) are affine, so they leave variance unchanged;
we map each final column back to its original counterpart and compare std / variance
directly.

**How to read it:**
- `std_ratio` (final / original) is the **practical** measure — values near `1.0` mean
  the variation is preserved; `<1` means the final set is more homogeneous (spread
  shrank), `>1` means it's more spread out.
- `levene_p` tests equality of variances, but with `n ≈ 6800` in the original pool it
  flags *tiny* differences as "significant" — so treat the effect size (`std_ratio`),
  not the p-value, as the headline.

In [ ]:
from scipy.stats import levene
import numpy as np
import pandas as pd

# (original column in leaders_df, corresponding column in filtered_final)
# affine renames are noted; variance is invariant to them.
dim_map = [
    ("v2xpa_antiplural", "v2xpa_plural"),       # 1 - x
    ("v2xpa_popul",      "v2xpa_antipopul"),    # 1 - x
    ("v2paimmig_osp",    "v2paimmig_osp"),
    ("v2palgbt_osp",     "v2palgbt_osp"),
    ("v2paculsup_osp",   "v2paculsup_osp"),
    ("v2parelig_osp",    "v2parelig_osp"),
    ("v2pawomlab_osp",   "v2pawomlab_osp"),
    ("v2paclient_osp",   "v2paanticlient_osp"), # 4 - x
    ("v2pariglef_osp",   "v2pariglef_osp"),
]

rows = []
for orig_col, fin_col in dim_map:
    orig = pd.to_numeric(leaders_df[orig_col], errors="coerce").dropna()
    fin  = pd.to_numeric(filtered_final[fin_col], errors="coerce").dropna()
    lev_stat, lev_p = levene(orig, fin, center="median")
    rows.append({
        "dimension":    orig_col,
        "n_original":   len(orig),
        "n_final":      len(fin),
        "std_original": orig.std(),
        "std_final":    fin.std(),
        "std_ratio":    fin.std() / orig.std(),
        "var_original": orig.var(),
        "var_final":    fin.var(),
        "levene_p":     lev_p,
        "diff_var?":    "yes" if lev_p < 0.05 else "no",
    })

res = pd.DataFrame(rows).set_index("dimension")
print(f"original pool: {len(leaders_df)} leaders   |   final set: {len(filtered_final)} leaders")
print(f"Dimensions with significantly different variance (Levene p<0.05): "
      f"{(res['levene_p'] < 0.05).sum()} / {len(res)}")
print(f"std_ratio  -> mean {res['std_ratio'].mean():.3f}, "
      f"range [{res['std_ratio'].min():.3f}, {res['std_ratio'].max():.3f}]")
res.round(4)

In [ ]:
article_length_cols = ["grokipedia_0_1", "grokipedia_0_2", "wikipedia"]

lengths = filtered_final[article_length_cols].fillna("").apply(
    lambda col: col.str.split().str.len()
)

length_summary = lengths.agg(["count", "mean", "median", "std", "min", "max"]).T
length_summary.columns = ["Articles", "Mean words", "Median words", "Std. dev.", "Min", "Max"]

display(length_summary.round(2))

lengths.plot(
    kind="box",
    figsize=(10, 6),
    showfliers=False,
    ylabel="Number of words",
    title="Article Length Comparison",
)
plt.xticks(range(1, 4), ["Grokipedia 0.1", "Grokipedia 0.2", "Wikipedia"])
plt.show()